In [1]:
import os
import json
import csv
import ast
import warnings
warnings.filterwarnings("ignore")

In [2]:
def text_to_bio(tok_list, obj):
  try:
    tok_list = [i.lower() for i in tok_list]
    bio_labels = ["O"] * len(tok_list)

    mapping = {"Participants": "PART", "Intervention":"INT", "Outcome":"OUT"}

    for i in mapping:
      for j in obj.get(i, []):
        entity_words = j.lower().split()
        for k in range (len(tok_list) - len(entity_words) + 1):
          if tok_list[k:k+len(entity_words)] == entity_words:
            bio_labels[k] = "B-" + mapping[i]
            for l in range(k+1, k+len(entity_words)):
              bio_labels[l] = "I-" + mapping[i]
    return bio_labels
  except Exception as e:
    print(f"Error in text_to_bio: {e}") # So you know which doc had issues
    try:
        return ["O"] * len(tok_list)
    except:
        return []



In [3]:
def config(df, file_path, doc_no, extract_fun ):
  if isinstance(df["Tokens"].iloc[0], str):
        df["Tokens"] = df["Tokens"].apply(ast.literal_eval)

  headers = ["Doc_id", "Document", "JSON", "Participants", "Intervention", "Outcome", "BIO Labels"]
  if not os.path.exists(file_path):
    with open(file_path, "w", newline = "") as f:
      writer  = csv.DictWriter(f, fieldnames = headers)
      writer.writeheader()


  for i in range(0, doc_no+1):
    document_id = df["Doc_id"][i]
    document_text = df["Documents"][i]
    tok_list = df["Tokens"][i]


    result = extract_fun(document_text)
    for category in ["Participants", "Intervention", "Outcome"]:
            raw_list = result.get(category, [])
            if isinstance(raw_list, list):
                clean_list = list(set(str(item).lower().strip() for item in raw_list))
                result[category] = clean_list
            else:
                result[category] = []

    obj = result
    Participants = obj.get("Participants", [])
    Intervention = obj.get("Intervention", [])
    Outcome = obj.get("Outcome", [])
    bio_labels = text_to_bio(tok_list, obj)

    with open(file_path, "a", newline = "") as f:
      writer = csv.DictWriter(f, fieldnames=headers)
      writer.writerow({
              "Doc_id": document_id,
              "Document": document_text,
              "JSON": result,
              "Participants": Participants,
              "Intervention": Intervention,
              "Outcome": Outcome,
              "BIO Labels": bio_labels
          })

      print(f"Finished and saved Doc ID: {document_id}")

In [ ]:
# import re # Make sure this is at the top of your utils file!

# def config(df, file_path, doc_no, extract_fun):
#     # Fix 1: Only eval tokens if they are strings
#     if isinstance(df["Tokens"].iloc[0], str):
#         df["Tokens"] = df["Tokens"].apply(ast.literal_eval)

#     headers = ["Doc_id", "Document", "JSON", "Participants", "Intervention", "Outcome", "BIO Labels"]

#     if not os.path.exists(file_path):
#         with open(file_path, "w", newline="") as f:
#             writer = csv.DictWriter(f, fieldnames=headers)
#             writer.writeheader()

#     for i in range(0, doc_no + 1):
#         document_id = df["Doc_id"][i]
#         document_text = df["Documents"][i]
#         tok_list = df["Tokens"][i]

#         result = extract_fun(document_text)

#         # Fix 2: Safe JSON Extraction logic
#         try:
#             # Look for everything between the first { and the last }
#             match = re.search(r'\{.*\}', result, re.DOTALL)
#             if match:
#                 json_str = match.group(0)
#                 obj = json.loads(json_str)
#             else:
#                 # If no JSON found, create a fallback empty dict
#                 obj = {"Participants": [], "Intervention": [], "Outcome": []}
#                 print(f"Warning: No JSON found in Doc {document_id}")
#         except Exception as e:
#             print(f"Error parsing JSON for Doc {document_id}: {e}")
#             obj = {"Participants": [], "Intervention": [], "Outcome": []}

#         # Fix 3: Ensure we have a dictionary so .get() works
#         Participants = obj.get("Participants", [])
#         Intervention = obj.get("Intervention", [])
#         Outcome = obj.get("Outcome", [])

#         bio_labels = text_to_bio(tok_list, obj)

#         with open(file_path, "a", newline="") as f:
#             writer = csv.DictWriter(f, fieldnames=headers)
#             writer.writerow({
#                 "Doc_id": document_id,
#                 "Document": document_text,
#                 "JSON": result,
#                 "Participants": Participants,
#                 "Intervention": Intervention,
#                 "Outcome": Outcome,
#                 "BIO Labels": bio_labels
#             })

#         print(f"Finished and saved Doc ID: {document_id}")